# SFT on Alpaca Dataset
Supervised fine-tuning of the pretrained SparkLLM (SimpleGPT) on Stanford Alpaca instruction-following data.

**Prerequisites:** Run `download_alpaca` → `tokenize_sft` → this notebook. Also needs the base checkpoint from `pre_train_mar23`.

**Key detail:** Loss is computed only on Response tokens (masked SFT). Data is loaded pre-tokenized from `datasets_sft/alpaca/`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tokenizers

In [ ]:
# ==================== CONFIGURATION ====================

# Model architecture (MUST match pretrain exactly)
BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64  # 20
FF_HIDDEN_DIM = 4 * EMBED_DIM  # 5120

# SFT hyperparameters
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4       # effective batch = 32
EPOCHS = 3
MAX_LR = 2e-5
MIN_LR = 1e-6
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

# Checkpoints
PRETRAIN_CKPT = os.path.join(CHECKPOINT_DIR, "gpt_medium_phase2.pth")
SFT_CKPT = os.path.join(CHECKPOINT_DIR, "gpt_medium_sft.pth")

# Hardware
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda")

In [ ]:
# ==================== MODEL DEFINITION ====================
# (Must match pretrain exactly for checkpoint loading)

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.norm1(x)), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)), p=self.dropout, training=self.training)
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList(
            [TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.1) for _ in range(NUM_LAYERS)]
        )
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

In [ ]:
# ==================== LOAD PRETRAINED CHECKPOINT ====================

with open(TOKENIZE_META, "r") as f:
    tok_meta = json.load(f)
VOCAB_SIZE = int(tok_meta["vocab_size"])
EOT_ID = int(tok_meta["eot_id"])

if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab padded to {VOCAB_SIZE} (Tensor Core optimized)")

model = SimpleGPT(VOCAB_SIZE).to(device)

if not os.path.exists(PRETRAIN_CKPT):
    raise FileNotFoundError(f"Pretrained checkpoint not found: {PRETRAIN_CKPT}")

print(f"Loading pretrained checkpoint: {PRETRAIN_CKPT}")
state_dict = torch.load(PRETRAIN_CKPT, map_location=device)
new_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model_state = model.state_dict()
safe_state = {k: v for k, v in new_state.items() if k in model_state and v.shape == model_state[k].shape}
model.load_state_dict(safe_state, strict=False)
print(f"Loaded {len(safe_state)}/{len(model_state)} layers")

print("Compiling model...")
model = torch.compile(model)

# Optimizer with lower LR for fine-tuning
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0},
]
optimizer = optim.AdamW(optim_groups, lr=MAX_LR, fused=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params / 1e6:.0f}M parameters")

In [ ]:
# ==================== ALPACA SFT DATASET ====================

class AlpacaSFTDataset(Dataset):
    """
    Loads pre-tokenized JSONL with input_ids and prefix_len.
    Builds masked labels: -100 for instruction tokens, actual ids for response tokens.
    """
    def __init__(self, tokenized_path, block_size, eot_id):
        self.block_size = block_size
        self.input_ids = []
        self.labels = []

        print(f"Loading pre-tokenized data from {tokenized_path}...")
        with open(tokenized_path, "r", encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)
                ids = ex["input_ids"]
                prefix_len = ex["prefix_len"]

                # Truncate to block_size + 1 (need +1 for shifted labels)
                if len(ids) > block_size + 1:
                    ids = ids[:block_size + 1]

                seq_len = len(ids) - 1
                input_ids = ids[:-1]
                target_ids = ids[1:]

                # Mask: positions 0..prefix_len-2 predict prefix tokens -> mask
                # Position prefix_len-1 predicts first response token -> keep
                labels = [-100] * seq_len
                for i in range(prefix_len - 1, seq_len):
                    labels[i] = target_ids[i]

                # Pad to block_size
                pad_len = block_size - len(input_ids)
                if pad_len > 0:
                    input_ids = input_ids + [eot_id] * pad_len
                    labels = labels + [-100] * pad_len

                self.input_ids.append(np.array(input_ids, dtype=np.int64))
                self.labels.append(np.array(labels, dtype=np.int64))

        print(f"Dataset: {len(self.input_ids):,} examples")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.input_ids[idx]),
            torch.from_numpy(self.labels[idx]),
        )

dataset = AlpacaSFTDataset(TOKENIZED_FILE, BLOCK_SIZE, EOT_ID)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)

print(f"Batches per epoch: {len(loader):,}")

In [ ]:
# ==================== TRAINING LOOP ====================

total_steps = (len(loader) * EPOCHS) // GRAD_ACCUM_STEPS
print(f"Total optimizer steps: {total_steps:,} | Warmup: {WARMUP_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

def get_lr(it, total_it):
    if it < WARMUP_STEPS:
        return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

curr_step = 0
best_loss = float('inf')
train_start = time.time()

print(f"\nSTARTING SFT (BFloat16, Masked Loss)\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_loss_count = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", dynamic_ncols=True, mininterval=1.0)

    for batch_idx, (xb, yb) in enumerate(pbar):
        lr = get_lr(curr_step, total_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(xb)
            loss = F.cross_entropy(
                logits.view(-1, VOCAB_SIZE),
                yb.view(-1),
                ignore_index=-100,
            )
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            curr_step += 1

            step_loss = loss.item() * GRAD_ACCUM_STEPS
            epoch_loss_sum += step_loss
            epoch_loss_count += 1
            elapsed = time.time() - train_start

            pbar.set_postfix(
                loss=f"{step_loss:.3f}",
                lr=f"{lr:.2e}",
                step=curr_step,
                mins=f"{elapsed/60:.1f}",
            )

    avg_loss = epoch_loss_sum / max(1, epoch_loss_count)
    print(f"Epoch {epoch} complete | avg_loss={avg_loss:.4f} | steps={curr_step}")

    # Save checkpoint after each epoch
    torch.save(model.state_dict(), SFT_CKPT)
    print(f"  Saved: {SFT_CKPT}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(CHECKPOINT_DIR, "gpt_medium_sft_alpaca_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"  New best! Saved: {best_path}")

total_time = time.time() - train_start
print(f"\nSFT Complete. {total_time/60:.1f} minutes, {curr_step} steps, best_loss={best_loss:.4f}")

In [ ]:
# ==================== TRAINING LOOP ====================

total_steps = (len(loader) * EPOCHS) // GRAD_ACCUM_STEPS
print(f"Total optimizer steps: {total_steps:,} | Warmup: {WARMUP_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

def get_lr(it, total_it):
    if it < WARMUP_STEPS:
        return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

curr_step = 0
best_loss = float('inf')
train_start = time.time()

print(f"\nSTARTING SFT (BFloat16, Masked Loss)\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_loss_count = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", dynamic_ncols=True, mininterval=1.0)

    for batch_idx, (xb, yb) in enumerate(pbar):
        lr = get_lr(curr_step, total_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(xb)
            loss = F.cross_entropy(
                logits.view(-1, VOCAB_SIZE),
                yb.view(-1),
                ignore_index=-100,
            )
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            curr_step += 1

            step_loss = loss.item() * GRAD_ACCUM_STEPS
            epoch_loss_sum += step_loss
            epoch_loss_count += 1
            elapsed = time.time() - train_start

            pbar.set_postfix(
                loss=f"{step_loss:.3f}",
                lr=f"{lr:.2e}",
                step=curr_step,
                mins=f"{elapsed/60:.1f}",
            )

    avg_loss = epoch_loss_sum / max(1, epoch_loss_count)
    print(f"Epoch {epoch} complete | avg_loss={avg_loss:.4f} | steps={curr_step}")

    # Save checkpoint after each epoch
    torch.save(model.state_dict(), SFT_CKPT)
    print(f"  Saved: {SFT_CKPT}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(CHECKPOINT_DIR, "gpt_medium_sft_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"  New best! Saved: {best_path}")

total_time = time.time() - train_start
print(f"\nSFT Complete. {total_time/60:.1f} minutes, {curr_step} steps, best_loss={best_loss:.4f}")

In [ ]:
# ==================== QUICK EVAL ====================

@torch.no_grad()
def generate(prompt_text, max_new_tokens=200, temperature=0.7, top_k=50):
    model.eval()
    ids = tokenizer.encode(prompt_text).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -BLOCK_SIZE:]  # crop to block size
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x_cond)
        logits = logits[:, -1, :] / temperature

        # Top-k filtering
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        if next_id.item() == EOT_ID:
            break

        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

# Test prompts
test_prompts = [
    "Instruction: Give three tips for staying healthy.\nResponse:",
    "Instruction: Explain the theory of relativity in simple terms.\nResponse:",
    "Instruction: Write a short poem about the ocean.\nResponse:",
]

for prompt in test_prompts:
    print(f"{'='*60}")
    result = generate(prompt)
    print(result)
    print()